In [ ]:
import os
import re
import json
import sqlite3
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

try:
    from google.colab import drive
    drive.mount('/content/drive')
    ruta_base = "/content/drive/My Drive/Entrega_2/"
except ImportError:
    ruta_base = "./"

ruta_salida = os.path.join(ruta_base, "salidas_entrega2")
os.makedirs(ruta_salida, exist_ok=True)

F_SUI   = ruta_base + "Presuntos_Suicidios_Colombia_2015_a_2024.xls"
F_POB_A = ruta_base + "DCD-area-sexo-edad-proypoblacion-dep-2005-2017_VP_2005-2017.xlsx"
F_POB_B = ruta_base + "DCD-area-sexo-edad-proypoblacion-dep-2005-2017_VP_2018-2050.xlsx"

for f in [F_SUI, F_POB_A, F_POB_B]:
    print(("OK      " if os.path.exists(f) else "NO EXISTE ") + os.path.basename(f))

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)

AZUL, ROJO, GRIS = "#1f3a5f", "#b8433a", "#8a8f98"
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

# Mismos grupos del INMLCF: 15-17 y 18-19 se mantienen separados (menores / mayores de edad).
# 00-04 no se incluye: no tiene casos y se excluye del denominador (población de 5 años y más).
GRUPOS_EDAD = ["05-09", "10-14", "15-17", "18-19", "20-24", "25-29", "30-34", "35-39",
               "40-44", "45-49", "50-54", "55-59", "60-64", "65-69", "70-74", "75-79", "80+"]
BORDES_EDAD = [5, 10, 15, 18, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 200]

In [ ]:
def extraer_casos(ruta):
    """Extrae la base de casos del INMLCF sin modificarla."""
    return pd.read_excel(ruta)

df_sui_raw = extraer_casos(F_SUI)
print(f"Registros: {df_sui_raw.shape[0]:,}  |  Columnas: {df_sui_raw.shape[1]}")
print(df_sui_raw.head())

In [ ]:
df_sui_raw.info()

In [ ]:
def extraer_poblacion_a(ruta):
    return pd.read_excel(ruta, header=11, dtype={"DP": str})

df_pob_a_raw = extraer_poblacion_a(F_POB_A)
print(f"Filas: {df_pob_a_raw.shape[0]:,}  |  Columnas: {df_pob_a_raw.shape[1]}")
print("Áreas:", df_pob_a_raw["ÁREA GEOGRÁFICA"].unique())
print(df_pob_a_raw.iloc[:5, :8])

In [ ]:
def extraer_poblacion_b(ruta):
    df = pd.read_excel(ruta, sheet_name="PobDepartamentalxÁreaSexoEdad", header=8, dtype={"Unnamed: 0": str})
    return df.rename(columns={"Unnamed: 0": "DP", "Unnamed: 1": "DPNOM", "Unnamed: 2": "AÑO", "Unnamed: 3": "ÁREA GEOGRÁFICA"})

df_pob_b_raw = extraer_poblacion_b(F_POB_B)
print(f"Filas: {df_pob_b_raw.shape[0]:,}  |  Columnas: {df_pob_b_raw.shape[1]}")
print("Áreas:", df_pob_b_raw["ÁREA GEOGRÁFICA"].dropna().unique())
print("\nÚltimas filas (notas al pie):")
print(df_pob_b_raw["DP"].tail(4).to_string())

In [ ]:
col = {c: c for c in df_sui_raw.columns}
C_DPTO_COD = [c for c in df_sui_raw.columns if "Departamento" in c and "digo" in c][0]
C_HORA     = [c for c in df_sui_raw.columns if "Rango de Hora" in c][0]
C_RAZON    = [c for c in df_sui_raw.columns if "Suicidio" in c and "Raz" in c][0]
C_ORIENT   = [c for c in df_sui_raw.columns if "Orientaci" in c][0]

df_anom = df_sui_raw.copy()
texto = df_anom.select_dtypes(include=["object", "string"])

#Texto mal codificado
df_anom["Texto_Mal_Codificado"] = texto.apply(lambda s: s.astype(str).str.contains("Ã")).any(axis=1)
#Caso sin departamento
df_anom["Sin_Departamento"] = df_anom[C_DPTO_COD] == 999
#Hora del hecho sin información
df_anom["Hora_Sin_Info"] = df_anom[C_HORA].astype(str).str.lower().str.contains("sin inform")
# Razón del suicidio sin información
df_anom["Razon_Sin_Info"] = df_anom[C_RAZON].astype(str).str.lower().str.contains("sin inform")
#Variable no implementada en el año del registro
df_anom["Variable_No_Implementada"] = df_anom[C_ORIENT].astype(str).str.contains("implementada")
#IDs duplicados
df_anom["ID_Duplicado"] = df_anom.iloc[:, 0].duplicated(keep=False)

banderas = ["Texto_Mal_Codificado", "Sin_Departamento", "Hora_Sin_Info", "Razon_Sin_Info",
            "Variable_No_Implementada", "ID_Duplicado"]
resumen_banderas = pd.DataFrame({"registros": df_anom[banderas].sum(),
                                 "%": (100 * df_anom[banderas].mean()).round(1)})
celdas_mojibake = int(texto.apply(lambda s: s.astype(str).str.contains("Ã")).sum().sum())
cols_mojibake = sum("Ã" in c for c in df_sui_raw.columns)
print(f"Celdas con texto mal codificado: {celdas_mojibake:,}  |  columnas con nombre corrupto: {cols_mojibake} de {df_sui_raw.shape[1]}")
display(resumen_banderas)

In [ ]:
def corregir_codificacion(x):
    """Texto UTF-8 leído como Latin-1 (p. ej. 'BogotÃ¡' -> 'Bogotá')."""
    if not isinstance(x, str):
        return x
    x = x.replace("_x0089_", "\x89")
    try:
        return x.encode("latin-1").decode("utf-8")
    except (UnicodeEncodeError, UnicodeDecodeError):
        return x

def normalizar_etiqueta(x):
    """Minúsculas, sin tildes, sin espacios dobles ni espacio antes de '('."""
    if not isinstance(x, str):
        return x
    x = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode()
    x = re.sub(r"\s+\(", "(", x.strip().lower())
    return re.sub(r"\s+", " ", x)

print(corregir_codificacion("BogotÃ¡, D.C."), "|", normalizar_etiqueta("Soltero (a)"), "=", normalizar_etiqueta("Soltero(a)"))

In [ ]:
TOKENS_SIN_INFO = ["sin informacion", "no habia sido implementada", "no sabe / no informa", "por determinar"]

perfil = []
for c in df_sui_raw.columns:
    crudo = df_sui_raw[c]
    normal = crudo.map(corregir_codificacion).map(normalizar_etiqueta)
    es_texto = crudo.dtype == object or str(crudo.dtype).startswith("str")
    pct = normal.astype(str).apply(lambda v: any(t in v for t in TOKENS_SIN_INFO)).mean() if es_texto else 0.0
    perfil.append({"variable": corregir_codificacion(c).replace("﻿", ""),
                   "nulos": int(crudo.isna().sum()),
                   "categorias_crudas": crudo.nunique(),
                   "categorias_normalizadas": normal.nunique(),
                   "pct_sin_info": round(100 * pct, 1)})
perfil = pd.DataFrame(perfil)
perfil.to_csv(os.path.join(ruta_salida, "perfil_calidad_inmlcf.csv"), index=False)
display(perfil)

In [ ]:
# INMLCF
p = perfil[perfil["pct_sin_info"] > 0].sort_values("pct_sin_info")
q = perfil[(perfil["categorias_crudas"] > perfil["categorias_normalizadas"]) & (perfil["categorias_crudas"] < 200)].sort_values("categorias_crudas")
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].barh(p["variable"], p["pct_sin_info"], color=[ROJO if v > 30 else AZUL for v in p["pct_sin_info"]])
ax[0].set_title("% de registros sin información / no implementado"); ax[0].tick_params(axis="y", labelsize=7)
y = np.arange(len(q))
ax[1].barh(y + 0.2, q["categorias_crudas"], 0.4, color=GRIS, label="Crudas")
ax[1].barh(y - 0.2, q["categorias_normalizadas"], 0.4, color=AZUL, label="Tras normalizar texto")
ax[1].set_yticks(y); ax[1].set_yticklabels(q["variable"], fontsize=7)
ax[1].legend(frameon=False); ax[1].set_title("Categorías distintas por variable")
fig.tight_layout(); fig.savefig(os.path.join(ruta_salida, "fig5_calidad.png"), dpi=200)
plt.show()

In [ ]:
anio = df_sui_raw.iloc[:, 1]
fx = lambda s: s.map(corregir_codificacion)
C_MAYOR = [c for c in df_sui_raw.columns if "Mayor Menor" in c][0]
C_MES   = [c for c in df_sui_raw.columns if "Mes del hecho" in c][0]
C_DPTO  = [c for c in df_sui_raw.columns if "Departamento del hecho" in c][0]

print("Etiqueta de mayoría de edad por año:")
display(pd.crosstab(anio, fx(df_sui_raw[C_MAYOR])))
print("Mes escrito con mayúscula inicial, por año:")
display(pd.crosstab(anio, df_sui_raw[C_MES].str[0].str.isupper()))
print("Orientación sexual 'No había sido implementada', por año:")
display(pd.crosstab(anio, df_sui_raw[C_ORIENT].astype(str).str.contains("implementada")))
print("Nombres de Bogotá y Quindío por año:")
m = df_sui_raw[C_DPTO_COD].isin([11, 63])
display(pd.crosstab(fx(df_sui_raw.loc[m, C_DPTO]), anio[m]))

In [ ]:
def corregir_texto(df):
    df = df.copy()
    df.columns = [corregir_codificacion(c).replace("﻿", "") for c in df.columns]
    for c in df.select_dtypes(include=["object", "string"]).columns:
        df[c] = df[c].map(corregir_codificacion)
    return df.rename(columns={"Año del hecho": "anio", "Sexo de la victima": "sexo",
                              "Grupo de Edad Quinquenal": "edad_q",
                              "Código Dane Departamento": "cod_dpto",
                              "Departamento del hecho DANE": "dpto_nombre"})

df_sui = corregir_texto(df_sui_raw)
print("Celdas con 'Ã' después de corregir:",
      int(df_sui.select_dtypes(include=["object", "string"]).apply(lambda s: s.astype(str).str.contains("Ã")).sum().sum()))
print(df_sui[["anio", "sexo", "edad_q", "cod_dpto", "dpto_nombre"]].head())

In [ ]:
HOMOLOGACION = {
    "Grupo Mayor Menor de Edad": {
        "a) menor de edad(<18 anos)": "Menor de edad", "a) menores de edad(<18 anos)": "Menor de edad",
        "b) mayor de edad(>18 anos)": "Mayor de edad", "b) mayores de edad(>18 anos)": "Mayor de edad"},
    "Estado Civil": {
        "soltero(a)": "Soltero(a)", "casado(a)": "Casado(a)", "viudo(a)": "Viudo(a)",
        "union libre": "Unión libre", "separado(a), divorciado(a)": "Separado(a)/divorciado(a)",
        "sin informacion": "Sin información", "no aplica": "No aplica"},
    "Escolaridad": {
        "ninguna": "Sin escolaridad", "sin escolaridad": "Sin escolaridad",
        "preescolar": "Preescolar", "educacion inicial y educacion preescolar": "Preescolar",
        "basica primaria": "Primaria", "educacion basica primaria": "Primaria",
        "basica secundaria": "Secundaria", "educacion basica secundaria o secundaria baja": "Secundaria",
        "educacion media o secundaria alta": "Media",
        "tecnologica": "Técnica/tecnológica", "educacion tecnica profesional y tecnologica": "Técnica/tecnológica",
        "profesional": "Universitaria", "universitario": "Universitaria",
        "especializacion, maestria o equivalente": "Posgrado", "maestria": "Posgrado", "doctorado o equivalente": "Posgrado",
        "sin informacion": "Sin información"},
    "Zona del Hecho": {
        "cabecera municipal": "Cabecera municipal", "parte rural(vereda y campo)": "Rural disperso",
        "centro poblado(corregimiento, inspeccion de policia y caserio)": "Centro poblado",
        "sin informacion": "Sin información"},
    "Mecanismo Causal de la Lesión Fatal": {
        "cortocontundente": "Cortocontundente", "corto contundente": "Cortocontundente",
        "cortopunzante": "Cortopunzante", "corto punzante": "Cortopunzante",
        "agente o mecanismo explosivo": "Explosivo", "agentes y mecanismo explosivo": "Explosivo",
        "mecanismo o agente explosivo": "Explosivo"},
}

def homologar(df):
    df = df.copy()
    for c in ["Mes del hecho", "Dia del hecho"]:
        df[c] = df[c].map(normalizar_etiqueta)
    for c, mapa in HOMOLOGACION.items():
        n = df[c].map(normalizar_etiqueta)
        df[c] = n.map(mapa).fillna(df[c])
    return df

antes = {c: df_sui[c].nunique() for c in list(HOMOLOGACION) + ["Mes del hecho", "Dia del hecho"]}
df_sui = homologar(df_sui)
display(pd.DataFrame({"antes": antes,
                      "despues": {c: df_sui[c].nunique() for c in antes}}))
print(df_sui["Escolaridad"].value_counts())

In [ ]:
df_sui["cod_dpto"] = df_sui["cod_dpto"].astype(int).astype(str).str.zfill(2)
sin_dpto = int((df_sui["cod_dpto"] == "999").sum())
df_sui = df_sui[df_sui["cod_dpto"] != "999"].copy()
print(f"Casos excluidos sin departamento: {sin_dpto}  |  Casos que continúan: {len(df_sui):,}")
print("Departamentos con más de un nombre en el INMLCF:")
print(df_sui.groupby("cod_dpto")["dpto_nombre"].unique().loc[lambda s: s.str.len() > 1])

In [ ]:
def mapear_edad_inmlcf(g):
    g = str(g)
    if "80" in g and "m" in g:
        return "80+"
    lo, hi = re.findall(r"\d+", g)[:2]        # (15 a 17) -> 15-17, (18 a 19) -> 18-19
    return f"{int(lo):02d}-{int(hi):02d}"

df_sui["grupo_edad"] = df_sui["edad_q"].map(mapear_edad_inmlcf)
display(df_sui.groupby(["edad_q", "grupo_edad"]).size().rename("casos").reset_index())

In [ ]:
casos = (df_sui.groupby(["cod_dpto", "anio", "sexo", "grupo_edad"]).size()
         .rename("casos").reset_index())
print(f"Combinaciones con al menos un caso: {len(casos):,}  |  Total casos: {casos['casos'].sum():,}")
print(casos.head())

In [ ]:
def edad_a_grupo(a):
    """Edad simple del DANE -> mismo grupo que usa el INMLCF."""
    if a < 5:
        return "00-04"
    for limite, grupo in zip(BORDES_EDAD[1:], GRUPOS_EDAD):
        if a < limite:
            return grupo

def pivotear_poblacion(df, patron, anios, area="Total"):
    """Ancho -> largo para un archivo DANE. patron: regex con grupos (sexo)(edad)."""
    df = df[pd.to_numeric(df["AÑO"], errors="coerce").notna()].copy()
    df["AÑO"] = df["AÑO"].astype(int)
    df = df[(df["ÁREA GEOGRÁFICA"] == area) & df["AÑO"].between(*anios)]
    cols = [c for c in df.columns if re.match(patron, str(c))]
    largo = df.melt(id_vars=["DP", "DPNOM", "AÑO"], value_vars=cols, var_name="col", value_name="pob")
    partes = largo["col"].str.extract(patron)
    largo["sexo"] = partes[0].map({"Hombres": "Hombre", "Mujeres": "Mujer"})
    largo["edad"] = partes[1].astype(int)
    largo["DP"] = largo["DP"].astype(str).str.zfill(2)
    return largo[largo["DP"].str.fullmatch(r"\d{2}")]

PATRON_A = r"^(Hombres|Mujeres)_(\d+)$"
PATRON_B = r"^(Hombres|Mujeres) (\d+)"

pob_a = pivotear_poblacion(df_pob_a_raw, PATRON_A, (2015, 2017))
pob_b = pivotear_poblacion(df_pob_b_raw, PATRON_B, (2018, 2024))
print(f"Largo 2015-2017: {len(pob_a):,} filas  |  Largo 2018-2024: {len(pob_b):,} filas")
print(pob_b.head())

In [ ]:
# Se cruza por código y se toma el nombre del archivo más reciente
nombres_a = pob_a.groupby("DP")["DPNOM"].first()
nombres_b = pob_b.groupby("DP")["DPNOM"].first()
print("Departamentos con nombre distinto entre los dos archivos DANE:")
print(pd.DataFrame({"2005-2017": nombres_a, "2018-2050": nombres_b}).loc[lambda d: d.iloc[:, 0] != d.iloc[:, 1]])

NOMBRE_DPTO = nombres_b.replace({"Archipiélago de San Andrés, Providencia y Santa Catalina": "San Andrés y Providencia"})

pob_largo = pd.concat([pob_a, pob_b], ignore_index=True)
pob_largo["grupo_edad"] = pob_largo["edad"].map(edad_a_grupo)
pob = (pob_largo.groupby(["DP", "AÑO", "sexo", "grupo_edad"])["pob"].sum().reset_index()
       .rename(columns={"DP": "cod_dpto", "AÑO": "anio"}))
pob["dpto"] = pob["cod_dpto"].map(NOMBRE_DPTO)
print(f"\nPoblación agrupada: {len(pob):,} filas (33 dptos x 10 años x 2 sexos x 18 grupos, incluye 00-04)")

In [ ]:
pob_dy = pob.groupby(["dpto", "anio"])["pob"].sum().unstack()
crec = (pob_dy.pct_change(axis=1) * 100).round(1)
crec["salto_vs_vecinos"] = (crec[2018] - crec[[2016, 2017, 2019, 2020]].mean(axis=1)).round(1)
display(crec[[2016, 2017, 2018, 2019, 2020, "salto_vs_vecinos"]].sort_values("salto_vs_vecinos", ascending=False).head(6))

tot = pob.groupby("anio")["pob"].sum()
print(f"Crecimiento nacional: 2016->17 {100*(tot[2017]/tot[2016]-1):.2f}% | 2017->18 {100*(tot[2018]/tot[2017]-1):.2f}% | 2018->19 {100*(tot[2019]/tot[2018]-1):.2f}%")
DPTOS_SALTO_2018 = ["25", "81", "95"]

In [ ]:
def integrar(pob, casos):
    pob = pob[pob["grupo_edad"] != "00-04"]        # denominador: población de 5 años y más
    df = pob.merge(casos, on=["cod_dpto", "anio", "sexo", "grupo_edad"], how="left")
    df["casos"] = df["casos"].fillna(0).astype(int)
    df["tasa_100k"] = df["casos"] / df["pob"] * 1e5
    return df

df = integrar(pob, casos)

sin_cruce = casos.merge(pob[["cod_dpto", "anio", "sexo", "grupo_edad"]], how="left", indicator=True)
print(f"Filas integradas: {len(df):,}")
print(f"Casos sin población (debe ser 0): {int(sin_cruce.loc[sin_cruce['_merge'] == 'left_only', 'casos'].sum())}")
print(f"Casos en el dataset integrado: {df['casos'].sum():,}")
print(df.head())

In [ ]:
std = df[df["anio"] == 2020].groupby(["sexo", "grupo_edad"])["pob"].sum()
pesos = std / std.sum()

tmp = df.groupby(["cod_dpto", "dpto", "sexo", "grupo_edad"])[["casos", "pob"]].sum().reset_index()
tmp["tasa"] = tmp["casos"] / tmp["pob"]
tmp["w"] = tmp.set_index(["sexo", "grupo_edad"]).index.map(pesos)

dep = (tmp.assign(tw=tmp["tasa"] * tmp["w"])
          .groupby(["cod_dpto", "dpto"])
          .agg(casos=("casos", "sum"), pob_total=("pob", "sum"), tasa_ajustada=("tw", "sum"))
          .reset_index())
dep["pob_media"] = dep["pob_total"] / 10
dep["tasa_cruda"] = dep["casos"] / dep["pob_total"] * 1e5
dep["tasa_ajustada"] = dep["tasa_ajustada"] * 1e5
dep["rank_casos"] = dep["casos"].rank(ascending=False, method="min").astype(int)
dep["rank_tasa"] = dep["tasa_ajustada"].rank(ascending=False, method="min").astype(int)
dep = dep.sort_values("tasa_ajustada", ascending=False).drop(columns="pob_total")
dep.to_csv(os.path.join(ruta_salida, "tasas_departamento.csv"), index=False)
display(dep.round(2))

In [ ]:
filas = []
for s, g in list(df.groupby("sexo")) + [("Total", df)]:
    y = g.groupby("anio")[["casos", "pob"]].sum()
    y["tasa"] = y["casos"] / y["pob"] * 1e5
    filas.append({"grupo": s, "casos": g["casos"].sum(),
                  "tasa_periodo": round(g["casos"].sum() / g["pob"].sum() * 1e5, 2),
                  "anio_min": f"{y['tasa'].idxmin()} ({y['tasa'].min():.2f})",
                  "anio_max": f"{y['tasa'].idxmax()} ({y['tasa'].max():.2f})"})
desc_sexo = pd.DataFrame(filas)
desc_sexo["%_casos"] = (100 * desc_sexo["casos"] / df["casos"].sum()).round(1)
display(desc_sexo)

tasa_h = df[df.sexo == "Hombre"]["casos"].sum() / df[df.sexo == "Hombre"]["pob"].sum()
tasa_m = df[df.sexo == "Mujer"]["casos"].sum() / df[df.sexo == "Mujer"]["pob"].sum()
print(f"Razón de tasas hombre/mujer: {tasa_h / tasa_m:.2f}")
print("\nTasa ajustada por departamento:")
print(dep["tasa_ajustada"].describe().round(2))

In [ ]:
nat = df.groupby("anio")[["casos", "pob"]].sum(); nat["tasa"] = nat["casos"] / nat["pob"] * 1e5
nat_sexo = df.groupby(["anio", "sexo"])[["casos", "pob"]].sum()
nat_sexo = (nat_sexo["casos"] / nat_sexo["pob"] * 1e5).unstack()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(nat.index, nat["casos"], color=AZUL)
for x, y in zip(nat.index, nat["casos"]):
    ax[0].text(x, y + 30, f"{y:,}".replace(",", "."), ha="center", fontsize=7)
ax[0].set_title("Casos de presunto suicidio por año"); ax[0].set_xticks(nat.index)
ax[1].plot(nat_sexo.index, nat_sexo["Hombre"], marker="o", color=AZUL, label="Hombres")
ax[1].plot(nat_sexo.index, nat_sexo["Mujer"], marker="o", color=ROJO, label="Mujeres")
ax[1].plot(nat.index, nat["tasa"], marker="o", color=GRIS, ls="--", label="Total")
ax[1].set_title("Tasa por 100.000 habitantes, por sexo"); ax[1].set_xticks(nat.index)
ax[1].legend(frameon=False, loc="center left", bbox_to_anchor=(0.0, 0.42))
fig.tight_layout(); fig.savefig(os.path.join(ruta_salida, "fig1_tendencia.png"), dpi=200)
plt.show()

In [ ]:
edad = df.groupby(["grupo_edad", "sexo"])[["casos", "pob"]].sum()
edad_t = (edad["casos"] / edad["pob"] * 1e5).unstack().reindex(GRUPOS_EDAD)

fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(len(GRUPOS_EDAD)); w = 0.4
ax.bar(x - w / 2, edad_t["Hombre"], w, color=AZUL, label="Hombres")
ax.bar(x + w / 2, edad_t["Mujer"], w, color=ROJO, label="Mujeres")
ax.set_xticks(x); ax.set_xticklabels(GRUPOS_EDAD, fontsize=8)
ax.set_ylabel("Tasa por 100.000"); ax.legend(frameon=False)
ax.set_title("Tasa de presunto suicidio por grupo de edad y sexo (2015–2024 acumulado)")
fig.tight_layout(); fig.savefig(os.path.join(ruta_salida, "fig2_edad_sexo.png"), dpi=200)
plt.show()
display(edad_t.round(2).T)

In [ ]:
d = dep.sort_values("tasa_ajustada")
fig, ax = plt.subplots(1, 2, figsize=(12, 7.5), sharey=True)
ax[0].barh(d["dpto"], d["casos"], color=GRIS); ax[0].set_title("Casos acumulados 2015–2024")
ax[1].barh(d["dpto"], d["tasa_ajustada"], color=AZUL)
ax[1].axvline(nat["tasa"].mean(), color=ROJO, ls="--", lw=1)
ax[1].set_title("Tasa ajustada por edad y sexo (x100.000)")
ax[0].tick_params(axis="y", labelsize=7)
fig.tight_layout(); fig.savefig(os.path.join(ruta_salida, "fig3_dptos.png"), dpi=200)
plt.show()
display(dep.query("rank_casos <= 3 or rank_tasa <= 3")[["dpto", "casos", "tasa_ajustada", "rank_casos", "rank_tasa"]].round(2))

In [ ]:
dy = df.groupby(["cod_dpto", "dpto", "anio"])[["casos", "pob"]].sum().reset_index()
dy["tasa"] = dy["casos"] / dy["pob"] * 1e5
cv = dy.groupby("dpto")["tasa"].agg(lambda s: s.std() / s.mean()).sort_values()

niveles = {
    "Dpto × año": dy["casos"],
    "Dpto × año × sexo": df.groupby(["cod_dpto", "anio", "sexo"])["casos"].sum(),
    "Dpto × año × edad": df.groupby(["cod_dpto", "anio", "grupo_edad"])["casos"].sum(),
    "Dpto × año × sexo × edad": df["casos"],
}
pct0 = {k: 100 * (v == 0).mean() for k, v in niveles.items()}

hm = dy.pivot_table(index="dpto", columns="anio", values="tasa").loc[d["dpto"][::-1]]
fig, ax = plt.subplots(1, 2, figsize=(12, 7.5), gridspec_kw={"width_ratios": [1.6, 1]})
im = ax[0].imshow(hm.values, aspect="auto", cmap="Blues")
ax[0].set_yticks(range(len(hm))); ax[0].set_yticklabels(hm.index, fontsize=7)
ax[0].set_xticks(range(10)); ax[0].set_xticklabels(hm.columns, fontsize=7, rotation=45)
ax[0].grid(False); ax[0].set_title("Tasa cruda por departamento y año")
fig.colorbar(im, ax=ax[0], fraction=0.04)
ax[1].barh(list(pct0)[::-1], list(pct0.values())[::-1], color=[ROJO, AZUL, AZUL, AZUL])
for i, v in enumerate(list(pct0.values())[::-1]):
    ax[1].text(v + 1, i, f"{v:.1f}%", va="center", fontsize=8)
ax[1].set_xlim(0, 100); ax[1].set_title("% de celdas con 0 casos\nsegún nivel de desagregación")
fig.tight_layout(); fig.savefig(os.path.join(ruta_salida, "fig4_heatmap_ceros.png"), dpi=200)
plt.show()

pequenos = dep.loc[dep["pob_media"] < 500_000, "cod_dpto"]
print("CV entre años (menor y mayor):"); print(cv.head(3).round(2).to_string()); print(cv.tail(4).round(2).to_string())
print(f"\n% celdas en cero: {pct0['Dpto × año × sexo × edad']:.1f}%  |  dptos < 500 mil hab ({len(pequenos)}): "
      f"{100*(df[df.cod_dpto.isin(pequenos)].casos == 0).mean():.1f}%  |  resto: {100*(df[~df.cod_dpto.isin(pequenos)].casos == 0).mean():.1f}%")
print(f"Celdas dpto-año con menos de 20 casos: {(dy['casos'] < 20).sum()} de {len(dy)}")
print(f"Casos en 0-9 años: {df[df.grupo_edad.isin(['00-04', '05-09'])]['casos'].sum()}")
vaupes_15_16 = dy[(dy["cod_dpto"] == "97") & (dy["anio"] <= 2016)]["casos"].sum()
print(f"Vaupés 2015-2016: {vaupes_15_16} casos")

In [ ]:
from scipy import stats

Q1, Q3 = dy["tasa"].quantile([0.25, 0.75])
IQR = Q3 - Q1
dy["Outlier_IQR"] = (dy["tasa"] < Q1 - 1.5 * IQR) | (dy["tasa"] > Q3 + 1.5 * IQR)
dy["Outlier_Zscore"] = np.abs(stats.zscore(dy["tasa"])) > 3

out = dy[dy["Outlier_IQR"] | dy["Outlier_Zscore"]].sort_values("tasa", ascending=False)
print(f"Límite superior IQR: {Q3 + 1.5 * IQR:.2f}  |  Outliers IQR: {dy['Outlier_IQR'].sum()}  |  Outliers Z-score: {dy['Outlier_Zscore'].sum()}")
display(out[["dpto", "anio", "casos", "pob", "tasa", "Outlier_IQR", "Outlier_Zscore"]].round(2))

In [ ]:
dy = dy.sort_values(["cod_dpto", "anio"])
g = dy.groupby("cod_dpto")
dy["casos_pm3"] = g["casos"].transform(lambda s: s.rolling(3, center=True, min_periods=2).sum())
dy["pob_pm3"] = g["pob"].transform(lambda s: s.rolling(3, center=True, min_periods=2).sum())
dy["tasa_pm3"] = dy["casos_pm3"] / dy["pob_pm3"] * 1e5

comp = dy.groupby("dpto").agg(cv_anual=("tasa", lambda s: s.std() / s.mean()),
                              cv_pm3=("tasa_pm3", lambda s: s.std() / s.mean())).round(2)
display(comp.sort_values("cv_anual", ascending=False).head(6))

fig, ax = plt.subplots(figsize=(12, 3.5))
for cod, color in [("97", ROJO), ("99", GRIS), ("05", AZUL)]:
    s = dy[dy.cod_dpto == cod]
    ax.plot(s["anio"], s["tasa"], color=color, alpha=0.35, marker="o")
    ax.plot(s["anio"], s["tasa_pm3"], color=color, lw=2, label=f"{s['dpto'].iloc[0]} (prom. móvil 3 años)")
ax.set_title("Tasa anual (tenue) frente a promedio móvil de 3 años"); ax.legend(frameon=False, fontsize=8)
plt.show()

In [ ]:
def pct_rural(df_raw, patron, anios):
    """% de población en 'Centros Poblados y Rural Disperso' por dpto-año."""
    rural = pivotear_poblacion(df_raw, patron, anios, area="Centros Poblados y Rural Disperso")
    total = pivotear_poblacion(df_raw, patron, anios, area="Total")
    r = rural.groupby(["DP", "AÑO"])["pob"].sum() / total.groupby(["DP", "AÑO"])["pob"].sum()
    return (100 * r).rename("pct_rural").reset_index().rename(columns={"DP": "cod_dpto", "AÑO": "anio"})

rural = pd.concat([pct_rural(df_pob_a_raw, PATRON_A, (2015, 2017)),
                   pct_rural(df_pob_b_raw, PATRON_B, (2018, 2024))])

comp_dy = df.groupby(["cod_dpto", "anio"]).apply(lambda g: pd.Series({
    "pob_total": g["pob"].sum(),
    "prop_hombres": g.loc[g.sexo == "Hombre", "pob"].sum() / g["pob"].sum(),
    "prop_15_29": g.loc[g.grupo_edad.isin(["15-17", "18-19", "20-24", "25-29"]), "pob"].sum() / g["pob"].sum(),
    "prop_60mas": g.loc[g.grupo_edad.isin(GRUPOS_EDAD[12:]), "pob"].sum() / g["pob"].sum(),
}), include_groups=False).reset_index().merge(rural, on=["cod_dpto", "anio"])

modelo = df[~df["grupo_edad"].isin(["00-04", "05-09"])].merge(comp_dy, on=["cod_dpto", "anio"])
modelo["anio_2020"] = (modelo["anio"] == 2020).astype(int)
modelo["flag_subregistro"] = ((modelo["cod_dpto"] == "97") & (modelo["anio"] <= 2016)).astype(int)
modelo["flag_salto_pob"] = (modelo["cod_dpto"].isin(DPTOS_SALTO_2018) & (modelo["anio"] == 2018)).astype(int)
modelo["log_pob"] = np.log(modelo["pob"])
modelo = pd.concat([modelo, pd.get_dummies(modelo[["sexo", "grupo_edad"]], prefix=["sexo", "edad"], dtype=int)], axis=1)

print(f"Dataset de modelado: {modelo.shape[0]:,} filas x {modelo.shape[1]} columnas")
print(f"% celdas en cero sin 0-9 años: {100 * (modelo['casos'] == 0).mean():.1f}%")
display(comp_dy.describe().round(3))
print(modelo.head())

In [ ]:
ruta_csv_int = os.path.join(ruta_salida, "dataset_integrado_dpto_anio_sexo_edad.csv")
ruta_csv_mod = os.path.join(ruta_salida, "dataset_modelado.csv")
df.to_csv(ruta_csv_int, index=False)
modelo.to_csv(ruta_csv_mod, index=False)

tamanos = {"CSV (sin compresión)": os.path.getsize(ruta_csv_mod) / 1024}
try:
    ruta_parquet = os.path.join(ruta_salida, "dataset_modelado.parquet")
    modelo.to_parquet(ruta_parquet, compression="snappy", index=False)
    tamanos["Parquet (Snappy)"] = os.path.getsize(ruta_parquet) / 1024
except ImportError:
    print("pyarrow no está instalado: se omite Parquet (en Colab ya viene instalado).")
display(pd.DataFrame({"Tamaño (KB)": tamanos}).round(1))

In [ ]:
REGION = {**dict.fromkeys(["08", "13", "20", "23", "44", "47", "70"], "Caribe"),
          **dict.fromkeys(["27", "19", "52", "76"], "Pacífica"),
          **dict.fromkeys(["05", "11", "15", "17", "25", "41", "54", "63", "66", "68", "73"], "Andina"),
          **dict.fromkeys(["50", "81", "85", "99"], "Orinoquía"),
          **dict.fromkeys(["18", "86", "91", "94", "95", "97"], "Amazonía"),
          "88": "Insular"}

def ciclo_vida(g):
    lo = 80 if g == "80+" else int(g[:2])
    return "Infancia" if lo < 10 else "Adolescencia" if lo < 15 else "Juventud" if lo < 30 else "Adultez" if lo < 60 else "Adulto mayor"

dim_departamento = pd.DataFrame({"cod_dpto": NOMBRE_DPTO.index, "nombre": NOMBRE_DPTO.values})
dim_departamento["region"] = dim_departamento["cod_dpto"].map(REGION)
dim_tiempo = pd.DataFrame({"anio": range(2015, 2025)})
dim_tiempo["es_2020"] = (dim_tiempo["anio"] == 2020).astype(int)
dim_tiempo["fuente_pob"] = np.where(dim_tiempo["anio"] <= 2017, "DANE 2005-2017", "DANE 2018-2050")
dim_demografia = pd.DataFrame([(s, g) for s in ["Hombre", "Mujer"] for g in GRUPOS_EDAD], columns=["sexo", "grupo_edad"])
dim_demografia.insert(0, "id_demografia", range(1, len(dim_demografia) + 1))
dim_demografia["ciclo_vida"] = dim_demografia["grupo_edad"].map(ciclo_vida)

hechos = df.merge(dim_demografia[["id_demografia", "sexo", "grupo_edad"]], on=["sexo", "grupo_edad"])
hechos["flag_subregistro"] = ((hechos["cod_dpto"] == "97") & (hechos["anio"] <= 2016)).astype(int)
hechos["flag_salto_pob"] = (hechos["cod_dpto"].isin(DPTOS_SALTO_2018) & (hechos["anio"] == 2018)).astype(int)
hechos = hechos[["cod_dpto", "anio", "id_demografia", "casos", "pob", "tasa_100k", "flag_subregistro", "flag_salto_pob"]]
print(dim_departamento.head()); print(dim_demografia.head())

In [ ]:
ruta_db = os.path.join(ruta_salida, "dw_suicidios.db")
conn_dw = sqlite3.connect(ruta_db)
cursor_dw = conn_dw.cursor()

cursor_dw.executescript("""
DROP TABLE IF EXISTS hechos_suicidio;
DROP TABLE IF EXISTS dim_departamento;
DROP TABLE IF EXISTS dim_tiempo;
DROP TABLE IF EXISTS dim_demografia;

CREATE TABLE dim_departamento (
  cod_dpto TEXT PRIMARY KEY,
  nombre   TEXT,
  region   TEXT
);
CREATE TABLE dim_tiempo (
  anio       INTEGER PRIMARY KEY,
  es_2020    INTEGER,
  fuente_pob TEXT
);
CREATE TABLE dim_demografia (
  id_demografia INTEGER PRIMARY KEY,
  sexo          TEXT,
  grupo_edad    TEXT,
  ciclo_vida    TEXT
);
CREATE TABLE hechos_suicidio (
  cod_dpto         TEXT,
  anio             INTEGER,
  id_demografia    INTEGER,
  casos            INTEGER,
  pob              REAL,
  tasa_100k        REAL,
  flag_subregistro INTEGER,
  flag_salto_pob   INTEGER,
  PRIMARY KEY (cod_dpto, anio, id_demografia),
  FOREIGN KEY (cod_dpto) REFERENCES dim_departamento(cod_dpto),
  FOREIGN KEY (anio) REFERENCES dim_tiempo(anio),
  FOREIGN KEY (id_demografia) REFERENCES dim_demografia(id_demografia)
);
""")
cursor_dw.execute("PRAGMA foreign_keys = ON;")

dim_departamento.to_sql("dim_departamento", conn_dw, if_exists="append", index=False)
dim_tiempo.to_sql("dim_tiempo", conn_dw, if_exists="append", index=False)
dim_demografia.to_sql("dim_demografia", conn_dw, if_exists="append", index=False)
hechos.to_sql("hechos_suicidio", conn_dw, if_exists="append", index=False)
conn_dw.commit()

for t in ["dim_departamento", "dim_tiempo", "dim_demografia", "hechos_suicidio"]:
    print(f"{t:18s} {pd.read_sql(f'SELECT COUNT(*) AS n FROM {t}', conn_dw)['n'][0]:>6,} filas")

In [ ]:
dup = pd.read_sql("""
SELECT cod_dpto, anio, id_demografia, COUNT(*) AS n
FROM hechos_suicidio GROUP BY cod_dpto, anio, id_demografia HAVING COUNT(*) > 1;
""", conn_dw)
print("1. Duplicados en la llave:", len(dup))

huerf = pd.read_sql("""
SELECT h.cod_dpto FROM hechos_suicidio h
LEFT JOIN dim_departamento d ON h.cod_dpto = d.cod_dpto
WHERE d.cod_dpto IS NULL;
""", conn_dw)
print("2. Hechos huérfanos:", len(huerf))

total_dw = pd.read_sql("SELECT SUM(casos) AS casos FROM hechos_suicidio", conn_dw)["casos"][0]
print(f"3. Casos en DW: {total_dw:,} | Casos fuente: {len(df_sui_raw):,} - {sin_dpto} sin dpto = {len(df_sui_raw) - sin_dpto:,} "
      f"-> {'OK' if total_dw == len(df_sui_raw) - sin_dpto else 'NO CUADRA'}")

print("4. Filas con población <= 0:", pd.read_sql("SELECT COUNT(*) AS n FROM hechos_suicidio WHERE pob <= 0", conn_dw)["n"][0])

In [ ]:
consulta = pd.read_sql("""
SELECT d.region,
       dm.sexo,
       SUM(h.casos)                        AS casos,
       ROUND(SUM(h.casos) * 1e5 / SUM(h.pob), 2) AS tasa_100k
FROM hechos_suicidio h
JOIN dim_departamento d ON h.cod_dpto = d.cod_dpto
JOIN dim_demografia  dm ON h.id_demografia = dm.id_demografia
GROUP BY d.region, dm.sexo
ORDER BY tasa_100k DESC;
""", conn_dw)
display(consulta)
conn_dw.close()

In [ ]:
try:
    from prefect import flow, task
except ImportError:
    print("Prefect no está instalado: se ejecuta el flujo sin orquestador.")
    def task(fn=None, **kw): return fn if fn else (lambda f: f)
    def flow(fn=None, **kw): return fn if fn else (lambda f: f)

@task
def extract():
    return extraer_casos(F_SUI), extraer_poblacion_a(F_POB_A), extraer_poblacion_b(F_POB_B)

@task
def transform(df_sui_raw, df_pob_a_raw, df_pob_b_raw):
    s = homologar(corregir_texto(df_sui_raw))
    s["cod_dpto"] = s["cod_dpto"].astype(int).astype(str).str.zfill(2)
    s = s[s["cod_dpto"] != "999"]
    s["grupo_edad"] = s["edad_q"].map(mapear_edad_inmlcf)
    c = s.groupby(["cod_dpto", "anio", "sexo", "grupo_edad"]).size().rename("casos").reset_index()
    p = pd.concat([pivotear_poblacion(df_pob_a_raw, PATRON_A, (2015, 2017)),
                   pivotear_poblacion(df_pob_b_raw, PATRON_B, (2018, 2024))])
    p["grupo_edad"] = p["edad"].map(edad_a_grupo)
    p = (p.groupby(["DP", "AÑO", "sexo", "grupo_edad"])["pob"].sum().reset_index()
          .rename(columns={"DP": "cod_dpto", "AÑO": "anio"}))
    p["dpto"] = p["cod_dpto"].map(NOMBRE_DPTO)
    return integrar(p, c)

@task
def load(df_final):
    ruta = os.path.join(ruta_salida, "dataset_integrado_flujo.csv")
    df_final.to_csv(ruta, index=False)
    return f"{len(df_final):,} filas y {df_final['casos'].sum():,} casos cargados en {os.path.basename(ruta)}"

@flow(name="ETL_Suicidios_Departamental")
def etl_pipeline():
    crudos = extract()
    df_final = transform(*crudos)
    print(load(df_final))

etl_pipeline()

In [ ]:
import shutil
shutil.make_archive(os.path.join(ruta_base, "resultados_entrega2"), "zip", ruta_salida)
print(sorted(os.listdir(ruta_salida)))
try:
    from google.colab import files
    files.download(os.path.join(ruta_base, "resultados_entrega2.zip"))
except ImportError:
    pass